In [ ]:
# ===============================================
# Data Cleaning Script - Monday Working Hours Dataset
# ===============================================
# Team: Bassem Gamal Mohamed – Dina Fawzy – Mohamed Farag – Pola Mokhtar – Ahmed Mamdouh – Mustafa Ibrahim
# Purpose: Clean and prepare network traffic data for analysis or ML
# ===============================================

import os
import pandas as pd
import numpy as np

# ---------------------------
# 1) File paths and settings
# ---------------------------
file_path = r"C:\Users\el handsia\Documents\Monday\Monday-WorkingHours.pcap_ISCX.csv"
output_path = r"C:\Users\el handsia\Documents\Monday\Monday-WorkingHours_cleaned.csv"

# thresholds
missing_thresh = 0.80          # drop columns with >80% missing values
numeric_convert_thresh = 0.50  # convert to numeric if at least 50% convertible

# ---------------------------
# 2) Load the dataset
# ---------------------------
if not os.path.exists(file_path):
    raise FileNotFoundError(f"File not found: {file_path}")

try:
    df = pd.read_csv(file_path, low_memory=False)
except UnicodeDecodeError:
    df = pd.read_csv(file_path, encoding="latin1", low_memory=False)

print(f"Data loaded successfully: {df.shape[0]} rows, {df.shape[1]} columns")


Data loaded successfully: 529918 rows, 85 columns


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Show first few rows
print("\n Sample Data (first 5 rows):")
display(df.head())



 Sample Data (first 5 rows):


,flow_id,source_ip,source_port,destination_ip,destination_port,protocol,timestamp,flow_duration,total_fwd_packets,total_backward_packets,...,act_data_pkt_fwd,min_seg_size_forward,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min
0,192.168.10.5-8.254.250.126-49188-80-6,8.254.250.126,80.0,192.168.10.5,49188.0,6.0,2017-03-07 08:55:00,4.0,2.0,0.0,...,1.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,192.168.10.14-8.253.185.121-49486-80-6,8.253.185.121,80.0,192.168.10.14,49486.0,6.0,2017-03-07 08:56:00,3.0,2.0,0.0,...,1.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,192.168.10.3-192.168.10.9-88-1031-6,192.168.10.9,1031.0,192.168.10.3,88.0,6.0,2017-03-07 08:56:00,609.0,7.0,4.0,...,5.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,192.168.10.3-192.168.10.9-88-1032-6,192.168.10.9,1032.0,192.168.10.3,88.0,6.0,2017-03-07 08:56:00,879.0,9.0,4.0,...,7.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10,192.168.10.3-192.168.10.9-88-1033-6,192.168.10.9,1033.0,192.168.10.3,88.0,6.0,2017-03-07 08:56:00,1160.0,9.0,6.0,...,7.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# Dataset information
print("\n Dataset Info:")
df.info()


 Dataset Info:
<class 'pandas.core.frame.DataFrame'>
Index: 249213 entries, 0 to 529916
Data columns (total 72 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   flow_id                      249213 non-null  object 
 1   source_ip                    249213 non-null  object 
 2   source_port                  249213 non-null  float32
 3   destination_ip               249213 non-null  object 
 4   destination_port             249213 non-null  float32
 5   protocol                     249213 non-null  float32
 6   timestamp                    249213 non-null  object 
 7   flow_duration                249213 non-null  float64
 8   total_fwd_packets            249213 non-null  float32
 9   total_backward_packets       249213 non-null  float32
 10  total_length_of_fwd_packets  249213 non-null  float32
 11  total_length_of_bwd_packets  249213 non-null  float32
 12  fwd_packet_length_max        249213 non-null  f

In [ ]:
# Summary statistics for numeric columns
print("\n Summary Statistics:")
display(df.describe())


 Summary Statistics:


,source_port,destination_port,protocol,flow_duration,total_fwd_packets,total_backward_packets,total_length_of_fwd_packets,total_length_of_bwd_packets,fwd_packet_length_max,fwd_packet_length_min,...,act_data_pkt_fwd,min_seg_size_forward,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min
count,249213.000000,249213.000000,249213.000000,2.492130e+05,249213.000000,249213.000000,2.492130e+05,2.492130e+05,249213.000000,249213.000000,...,249213.000000,2.492130e+05,2.492130e+05,2.492130e+05,2.492130e+05,2.492130e+05,2.492130e+05,2.492130e+05,249213.0,249213.0
mean,46690.953125,1950.323242,11.186680,1.569772e+07,12.791824,14.150831,8.510822e+02,1.927957e+04,322.490875,19.908621,...,8.550088,-7.715274e+03,1.134126e+05,6.791406e+04,2.368459e+05,7.753350e+04,3.845092e+06,2.543634e+05,4033669.0,3588391.5
std,17616.619141,9683.250000,5.500256,3.506270e+07,693.593750,885.739746,5.785774e+03,2.151072e+06,567.302490,42.776028,...,650.750793,8.058338e+05,7.187430e+05,4.252076e+05,1.133862e+06,6.522808e+05,1.165548e+07,2.221309e+06,12164004.0,11355358.0
min,0.000000,0.000000,0.000000,0.000000e+00,1.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,...,0.000000,-8.388531e+07,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.0
25%,41837.000000,53.000000,6.000000,3.080000e+02,2.000000,2.000000,6.800000e+01,1.600000e+02,35.000000,0.000000,...,1.000000,2.000000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.0
50%,52687.000000,80.000000,6.000000,8.872200e+04,3.000000,2.000000,1.040000e+02,3.020000e+02,47.000000,0.000000,...,1.000000,3.200000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.0
75%,58199.000000,443.000000,17.000000,5.544972e+06,10.000000,8.000000,8.450000e+02,3.428000e+03,459.000000,37.000000,...,5.000000,3.200000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.0
max,65535.000000,65530.000000,17.000000,1.199996e+08,218658.000000,291260.000000,1.286884e+06,6.410000e+08,23360.000000,1472.000000,...,207501.000000,1.260000e+02,6.400000e+07,6.430000e+07,9.110000e+07,6.400000e+07,1.200000e+08,6.790000e+07,120000000.0,120000000.0


In [ ]:
# ---------------------------
# 3) Clean and normalize column names
# ---------------------------
df.columns = (
    df.columns
      .str.strip()
      .str.replace(' ', '_')
      .str.replace(r'[^\w]', '', regex=True)
      .str.lower()
)
print("\nColumn names normalized.")


Column names normalized.


In [ ]:
# ---------------------------
# 4) Quick overview
# ---------------------------
print("\nData types:")
print(df.dtypes.value_counts())
print("\nFirst few rows:")
print(df.head(3))

if "label" in df.columns:
    print("\nLabel distribution:")
    print(df["label"].astype(str).value_counts())


Data types:
int64      43
float64    37
object      5
Name: count, dtype: int64

First few rows:
                                 flow_id      source_ip  source_port  \
0  192.168.10.5-8.254.250.126-49188-80-6  8.254.250.126           80   
1  192.168.10.5-8.254.250.126-49188-80-6  8.254.250.126           80   
2  192.168.10.5-8.254.250.126-49188-80-6  8.254.250.126           80   

  destination_ip  destination_port  protocol      timestamp  flow_duration  \
0   192.168.10.5             49188         6  3/7/2017 8:55              4   
1   192.168.10.5             49188         6  3/7/2017 8:55              1   
2   192.168.10.5             49188         6  3/7/2017 8:55              1   

   total_fwd_packets  total_backward_packets  ...  min_seg_size_forward  \
0                  2                       0  ...                    20   
1                  2                       0  ...                    20   
2                  2                       0  ...                    20   


In [ ]:
# 5) Handle missing and infinite values
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Replace common string placeholders for missing data
df.replace(
    {
        "": np.nan,
        " ": np.nan,
        "nan": np.nan,
        "NaN": np.nan,
        "None": np.nan,
        "none": np.nan
    },
    inplace=True
)

In [ ]:
# ---------------------------
# 6) Drop columns with too many or all missing values
# ---------------------------

# Calculate the fraction of missing values for each column
missing_frac = df.isna().mean()

# Drop columns with missing values above the threshold
cols_to_drop = missing_frac[missing_frac > missing_thresh].index
df.drop(columns=cols_to_drop, inplace=True)

# Drop columns that are completely empty
df.dropna(axis=1, how='all', inplace=True)

# Display summary
print(f"Dropped {len(cols_to_drop)} columns with more than {missing_thresh*100:.0f}% missing values.")
print(f"Remaining columns: {df.shape[1]}")



Dropped 0 columns with more than 80% missing values.
Remaining columns: 72


In [ ]:
# ---------------------------
# 7) Convert object columns to numeric when possible
# ---------------------------

for col in df.select_dtypes(include="object"):
    temp = pd.to_numeric(df[col], errors="coerce")
    if temp.notna().mean() >= 0.9:
        df[col] = temp
        print(f"Converted '{col}' to numeric.")

In [ ]:
# ---------------------------
# 8) Handle missing values (Imputation)
# ---------------------------

# Numeric columns → fill with median
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].apply(lambda x: x.fillna(x.median()))

# Categorical columns → fill with "unknown" and remove extra spaces
cat_cols = df.select_dtypes(exclude=[np.number]).columns
df[cat_cols] = df[cat_cols].apply(lambda x: x.fillna("unknown").astype(str).str.strip())

# Summary
missing_after = df.isna().sum().sum()
if missing_after == 0:
    print(" All missing values have been handled successfully.")
else:
    print(f" {missing_after} missing values remain.")


 All missing values have been handled successfully.


In [ ]:
# ---------------------------
# 9) Remove constant and near-constant columns
# ---------------------------
const_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
near_const = [c for c in df.columns if df[c].value_counts(normalize=True, dropna=False).iloc[0] > 0.999 and c != "label"]

df.drop(columns=const_cols + near_const, inplace=True, errors="ignore")

print("Dropped:", const_cols + near_const if const_cols or near_const else "None")


Dropped: ['fin_flag_count']


In [ ]:
# ---------------------------
# 10) Drop duplicate rows
# ---------------------------
before = len(df)
df.drop_duplicates(subset=['Flow ID'] if 'Flow ID' in df.columns else None, inplace=True)
print(f"Dropped {before - len(df)} duplicate rows.")



Dropped 0 duplicate rows.


In [ ]:
# ---------------------------
# 11) Optimize numeric dtypes
# ---------------------------
for col in df.select_dtypes(include=[np.number]):
    df[col] = pd.to_numeric(df[col], downcast="float")

print("\nNumeric columns downcasted to save memory.")



Numeric columns downcasted to save memory.


In [ ]:
# ---------------------------
# 12) Final checks
# ---------------------------
print("\nFinal dataset shape:", df.shape)
print("Total missing values:", df.isna().sum().sum())

if "label" in df.columns:
    print("\nFinal label distribution:")
    print(df["label"].value_counts())



Final dataset shape: (249213, 71)
Total missing values: 0


In [ ]:
# ---------------------------
# 13) Save cleaned dataset
# ---------------------------
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df.to_csv(output_path, index=False)
print("\nCleaned dataset saved to:", output_path)


Cleaned dataset saved to: C:\Users\el handsia\Documents\Monday\Monday-WorkingHours_cleaned.csv
